## 1. Import des bibliothèques et chargement des données

In [30]:
import import_ipynb
import numpy as np
import pandas as pd
from adultdata_preparation_evaluat import prepare_adult_dataset

x_train, x_test, y_train, y_test, mapping = prepare_adult_dataset()

## 2. Définition de la fonction d'entraînement BRCG

In [ ]:
from aix360.algorithms.rbm import FeatureBinarizer, BooleanRuleCG
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

def train_brcg(x_train, y_train, x_test, y_test, lambda0=0.001, lambda1=0.001):
    
    # Binarisationa
    fb = FeatureBinarizer(negations=True)
    X_train_bin = fb.fit_transform(x_train)
    X_test_bin = fb.transform(x_test)

    # Nettoyage
    X_train_bin = np.nan_to_num(X_train_bin)
    X_test_bin = np.nan_to_num(X_test_bin)

    # Colonnes fixes
    columns = [f"f{i}" for i in range(X_train_bin.shape[1])]

    X_train_bin = pd.DataFrame(X_train_bin, columns=columns).astype(int)
    X_test_bin = pd.DataFrame(X_test_bin, columns=columns).astype(int)

    # Création du modèle
    brcg = BooleanRuleCG(lambda0=lambda0, lambda1=lambda1, verbose=False)

    start_time = time.time()
    brcg.fit(X_train_bin, y_train)
    train_time = time.time() - start_time

    # Prédiction
    y_pred = brcg.predict(X_test_bin)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, average='weighted'),
        "precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "train_time": train_time
    }

    try:
        rules = brcg.explain()
    except:
        rules = None

    
    return brcg, fb, columns, train_time, metrics, rules

## 3. Entrainement du modèle

In [32]:
# Entrainement
brcg_model, fb, columns, train_time, brcg_metrics, brcg_rules = train_brcg(
    x_train, y_train, x_test, y_test
)

# Affichage
print("Résultats BRCG")
print("Accuracy:", brcg_metrics["accuracy"])
print("F1-score:", brcg_metrics["f1"])
print("Precision:", brcg_metrics["precision"])
print("Recall:", brcg_metrics["recall"])

# Règles
if brcg_rules is not None:
    print("Règles générées :")
    print(brcg_rules)
else:
    print("Impossible d'afficher les règles")

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved
Iteration: 1, Objective: 0.2181
Iteration: 2, Objective: 0.1897
Iteration: 3, Objective: 0.1854
Iteration: 4, Objective: 0.1843
Iteration: 5, Objective: 0.1843
Iteration: 6, Objective: 0.1843
Résultats BRCG
Accuracy: 0.8189774297558728
F1-score: 0.8026137894465102
Precision: 0.8063169426792735
Recall: 0.8189774297558728
Impossible d'afficher les règles


## 4. Sauvegarde des résultats et des règles générées

In [ ]:
import pandas as pd
import os
# Créer dossier adultdataset 
os.makedirs("results/adultdataset", exist_ok=True)

# BRCG
brcg_df = pd.DataFrame([brcg_metrics])
brcg_df["method"] = "BRCG"

brcg_df.to_csv("results/adultdataset/brcg_metrics.csv", index=False)
# BRCG RULES
with open("results/adultdataset/brcg_rules.txt", "w") as f:
    f.write(str(brcg_rules))

## 5. Fonction d'évaluation détaillée du modèle BRCG

In [34]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_brcg(model, fb, X_train, y_train, X_test, y_test, elapsed=None, ram=None, columns=None):

    # Binarisation
    X_train_bin = fb.transform(X_train)
    X_test_bin = fb.transform(X_test)

    X_train_bin = np.nan_to_num(X_train_bin)
    X_test_bin = np.nan_to_num(X_test_bin)

    # Recréer DataFrame
    X_train_bin = pd.DataFrame(X_train_bin, columns=columns).astype(int)
    X_test_bin = pd.DataFrame(X_test_bin, columns=columns).astype(int)
    
    #predictions
    y_train_pred = model.predict(X_train_bin)
    y_test_pred = model.predict(X_test_bin)

    # Métriques
    f1_train = f1_score(y_train, y_train_pred, average="weighted")
    f1_test = f1_score(y_test, y_test_pred, average="weighted")

    exact_train = accuracy_score(y_train, y_train_pred)
    exact_test = accuracy_score(y_test, y_test_pred)

    # Complexité (nombre de règles)
    complexity = len(model.rules_) if hasattr(model, "rules_") else 1

    return {
        "F1_train": f1_train,
        "F1_test": f1_test,
        "Train_Coverage": np.nan,
        "Train_Exactitude": exact_train,
        "Test_Coverage": np.nan,
        "Test_Exactitude": exact_test,
        "Conflit": np.nan,
        "Non_couverts": np.nan,
        "F1_rmaj": np.nan,
        "F1_rr": f1_test,
        "Complexité": complexity,
        "Temps": round(elapsed, 2) if elapsed is not None else None,
        "RAM_MB": round(ram, 2) if ram is not None else None
    }